In [ ]:
!pip install braindecode[hub]==1.8.1
import braindecode

In [ ]:
import os
import importlib
import sys
from pathlib import Path
import json

from tqdm import tqdm

import numpy as np
import matplotlib
from matplotlib import pyplot as plt

from sklearn.manifold import TSNE

import h5py
import cv2

In [ ]:
sys.path.append("../src")
import emd_loader
import labram

importlib.reload(emd_loader)
importlib.reload(labram)

# Loading data

In [ ]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)
emd_dir = Path("/content/drive/MyDrive/eeg_moments_dataset")

In [ ]:
X = []
Y = []
for ses_id in tqdm(range(1,6+1)):
    dataset = emd_loader.EmdDataset(emd_dir)
    # X: (n_trials, n_channels, n_times)
    tmp_X, ch_names, tmp_Y = dataset.get_data_by_ids(1, 1)
    res = labram.labram_embed(tmp_X, 500, ch_names, batch_size = 32, device = "cuda")
    X.append(tmp_X)
    Y.append(tmp_Y)
X = np.concatenate(X)
Y = np.concatenate(Y)
print(X.shape)
print(Y.shape)

In [ ]:
def tsne_visualize(X, y, title):
    X_2d = TSNE(n_components=2 ,perplexity=30, random_state=42).fit_transform(X)
    classes = np.unique(y)
    for cls in classes:
        idx = y == cls
        plt.scatter(
            X_2d[idx, 0],
            X_2d[idx, 1],
            s=7,
        )
    plt.title(title)
tsne_visualize(res, Y, "LaBraM")